# RutaPro · Notebook 06: Robustez II. Réplicas del simulador, sensibilidad de supuestos y corte global

**La pregunta de fondo.** Todos los resultados del v1 salen de **una sola realización** de un simulador cuyas reglas definió el propio equipo. Ante un jurado, la objeción más fuerte es: *"¿y si el simulador hubiera sido distinto?"*. Este notebook la responde con evidencia en lugar de con argumentos.

| Sección | Qué hace | Qué problema del v1 ataca |
|---|---|---|
| 1 | Define 6 **escenarios** del simulador y muestra indicadores de plausibilidad de cada uno | Transparencia de las reglas y plausibilidad del embudo |
| 2 | Genera **10 datasets independientes por escenario** y repite todo el pipeline (ajuste en validación incluido) | La variación debida a los datos (no solo a la semilla del modelo) |
| 3 | Usa el **corte temporal global como protocolo principal** (con 400 estudiantes y calendario ampliado quedan ≈ 180 evaluables) | La fuga entre usuarios del esquema por usuario |
| 4 | Compara el esquema por usuario contra el global **sobre los mismos datos** | Cuantifica cuánto distorsiona la fuga |

**Escenarios** (todos con 400 estudiantes, registros en 60 días y ~20 días entre cursos):

| Escenario | Qué cambia respecto del v1 | Qué pone a prueba |
|---|---|---|
| E0 Base | Nada (reglas del v1) | Referencia a mayor escala |
| E1 Perfil débil | Afinidad 2.0→1.0, interés 0.6→0.3, exploración 15 %→30 % | ¿Sigue ganando el perfil si explica menos? |
| E2 Popularidad dominante | Popularidad 1.5→3.0, afinidad 1.0 | ¿Gana la popularidad si todos eligen lo masivo? |
| E3 Ruta secuencial | La siguiente elección depende del curso anterior (ruta de aprendizaje típica) | ¿Aparece valor en el historial? |
| E4 Gustos latentes | Cada estudiante tiene gustos individuales **no visibles** en sus atributos | ¿Ayudan los métodos colaborativos? |
| E5 Embudo MOOC bajo | Inscripción 10 a 30 % y finalización ~15 a 40 % (más cercano a lo reportado en MOOCs) | ¿Depende la conclusión de un embudo optimista? |

> Los datos siguen siendo **simulados**. El objetivo no es probar que el sistema funcionará con estudiantes reales, sino mostrar **bajo qué supuestos** la conclusión se mantiene y bajo cuáles no.

In [1]:
import os, sys, subprocess
from google.colab import drive

NECESITA_LIGHTFM = True

# 1) Conectar el Drive y entrar a la carpeta del proyecto
drive.mount('/content/drive')
PROY = "/content/drive/MyDrive/RutaPro"
os.chdir(PROY + "/02_Notebooks")
print("Carpeta actual:", os.getcwd())

# 2) Comprobar que están los archivos imprescindibles
faltan = [f for f in ["rutapro_rec.py", "rutapro_sim.py", "rutapro_extra.py", "rutapro_mooc.py",
                      "resultados/hiperparametros.json", "../01_Datos/cursos.csv",
                      "../01_Datos/pares_usuario_curso.csv"] if not os.path.exists(f)]
print("Archivos que faltan:", faltan if faltan else "ninguno, todo en su sitio")

# 3) LightFM: usa la copia compilada en el Drive; si no existe, la compila (≈ 2 min) y la guarda
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:], r.stderr[-1500:])
        raise RuntimeError("Falló: " + cmd[:70])

if NECESITA_LIGHTFM:
    LF = PROY + "/lightfm_src"
    if not os.path.exists(LF + "/lightfm"):
        print("Compilando LightFM, un momento...")
        sh("pip install -q cython")
        sh("rm -rf /content/lightfm_src && git clone -q --depth 1 https://github.com/lyst/lightfm.git /content/lightfm_src")
        p = "/content/lightfm_src/setup.py"
        s = open(p).read()
        s = s.replace("compiler_directives={'language_level' : \"3\"}",
                      "compiler_directives={'language_level': \"3\", 'legacy_implicit_noexcept': True}").replace("-march=native", "-O2")
        open(p, "w").write(s)
        sh("cd /content/lightfm_src && python setup.py cythonize -q && python setup.py build_ext --inplace -q")
        sh("rm -rf '" + LF + "' && cp -r /content/lightfm_src '" + LF + "'")
    sys.path.insert(0, LF)
    import lightfm
    print("LightFM", lightfm.__version__)

Mounted at /content/drive
Carpeta actual: /content/drive/MyDrive/RutaPro/02_Notebooks
Archivos que faltan: ninguno, todo en su sitio
LightFM 1.17


In [ ]:
try:
    import lightfm
    print("LightFM", lightfm.__version__)
except ImportError as e:
    raise SystemExit("Falta LightFM: instala con 'conda install -c conda-forge lightfm' (ver 02_Notebooks/README).") from e

In [ ]:
import json, time, warnings
from dataclasses import asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import stats
warnings.filterwarnings("ignore")

import rutapro_rec as rr
import rutapro_sim as sim
import rutapro_extra as ex

AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, GRIS = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#9a9890"
TXT, TXT2 = "#0b0b0b", "#52514e"
mpl.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#d4d3cd", "axes.labelcolor": TXT2, "xtick.color": TXT2, "ytick.color": TXT2,
    "text.color": TXT, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#ebeae6",
    "axes.axisbelow": True, "font.size": 10, "figure.dpi": 110,
})
BASE = rr.ruta_base()
DATA = BASE / "01_Datos"
FIG = BASE / "04_Informe" / "figuras"; FIG.mkdir(parents=True, exist_ok=True)
RES = BASE / "02_Notebooks" / "resultados_robustez"; RES.mkdir(parents=True, exist_ok=True)
def guardar(fig, n): fig.savefig(FIG / f"{n}.png", dpi=160, bbox_inches="tight")

cursos = pd.read_csv(DATA / "cursos.csv")
R = 10                                   # réplicas independientes por escenario
ESC = sim.escenarios()                   # 400 estudiantes, registros en 60 días, ~20 días entre cursos
print("Escenarios:", list(ESC))

## 1. Escenarios y plausibilidad

Parámetros de cada escenario y indicadores del dataset que genera (una muestra por escenario). Los indicadores permiten juzgar la **plausibilidad** del patrón, que el enunciado pide discutir explícitamente.

Referencias externas para contrastar el embudo: la literatura sobre MOOC reporta tasas de finalización bajas, del orden de 5 a 15 % de los inscritos (por ejemplo, Jordan, 2014, *International Review of Research in Open and Distributed Learning*; **verificar la cita y la cifra antes de incluirla en el informe**). RutaPro es una plataforma curada y con estudiantes motivados por su objetivo laboral, así que una finalización mayor es defendible, pero conviene mostrar que la conclusión no depende de ello (escenario E5).

In [ ]:
params = pd.DataFrame({n: asdict(P) for n, P in ESC.items()}).drop(index=["nombre", "seed"])
display(params)
desc = {}
for n, P in ESC.items():
    u, i, _ = sim.generar_dataset(cursos, P.con(seed=2000))
    pr = ex.construir_pares(i)
    d = ex.descriptivos_dataset(u, i, cursos)
    for esq, col in [("global", "particion_global"), ("usuario", "particion")]:
        tr = pr[pr[col].isin(["train", "valid"])]; te = pr[(pr[col] == "test") & (pr.positivo == 1)]
        d[f"evaluables ({esq})"] = len(set(te.user_id) & set(tr[tr.positivo == 1].user_id))
    desc[n] = d
desc = pd.DataFrame(desc)
desc.to_csv(RES / "plausibilidad_escenarios.csv")
display(desc.round(2))

**Interpretación.**
- **E0 (reglas del v1 a mayor escala)** reproduce los rasgos del v1: 53.5 % de las vistas terminan en inscripción, 50.6 % de las inscripciones en finalización, 5.3 inscripciones por estudiante, Gini de inscripciones por curso de 0.43 (0.41 en el v1) y 12 % de las inscripciones de quienes solo estudian en español en cursos en inglés (13 % en el v1).
- **Embudo.** La finalización de ~50 % es alta frente a lo que suele reportarse en MOOC (verificar cifra y cita). El escenario **E5** la baja a ~25 % y la conversión vista→inscripción a 24 %; sigue por encima del rango citado, pero en la dirección conservadora. Con ese embudo hay 2.75 inscripciones por estudiante y quedan ≈ 127 estudiantes evaluables con corte global (frente a 165 a 190 en los demás escenarios).
- **Concentración.** El Gini va de 0.30 (E4, gustos individuales dispersan la demanda) a 0.53 (E2, popularidad dominante), un rango que cubre el 0.41 observado en el v1.
- **Tamaño de evaluación.** Con el corte global quedan entre 127 y 183 estudiantes evaluables por dataset (30 en el v1), así que las comparaciones sin fuga entre usuarios ya tienen potencia razonable.

## 2. Réplicas por escenario

Para cada escenario:
1. Se ajustan los hiperparámetros **solo en validación** con una rejilla pequeña y la misma regla de parsimonia del notebook 02 (en un dataset de ajuste con otra semilla, para no ajustar sobre los datos de evaluación).
2. Se generan **10 datasets independientes** (semillas distintas del simulador).
3. En cada uno se entrena con entrenamiento + validación y se evalúa **una vez** en prueba, con el corte **global** (protocolo principal) y con el corte **por usuario** (comparación).
4. Modelos: Popularidad, Popularidad por segmento + reglas, Contenido, KNN usuario, KNN ítem, LightFM, LightFM + reglas y el Oráculo.

La ejecución tarda varios minutos.

In [ ]:
partes, hp_por_escenario = [], {}
t_total = time.time()
for nombre, P0 in ESC.items():
    # Cada escenario se guarda en _cache: si ya existe se reutiliza (borra la carpeta para recalcular todo)
    res_e, dif_e, hp = ex.correr_escenario(cursos, nombre, P0, R=R, cache=RES / "_cache" / f"{nombre.split()[0]}.pkl")
    partes.append((res_e, dif_e)); hp_por_escenario[nombre] = hp
print(f"Total: {(time.time() - t_total) / 60:.1f} min")
filas = pd.concat([p[0] for p in partes]); dif = pd.concat([p[1] for p in partes])
resultados = filas.reset_index(drop=True); diferencias = dif.reset_index(drop=True)
resultados.to_csv(RES / "replicas_resultados.csv", index=False); diferencias.to_csv(RES / "replicas_diferencias.csv", index=False)
json.dump(hp_por_escenario, open(RES / "replicas_hiperparametros.json", "w"), indent=1, ensure_ascii=False)

## 3. Resultados con el corte global (protocolo principal)

NDCG@10 medio de las 10 réplicas ± desviación entre réplicas (variación debida a los **datos**), y qué fracción del techo del oráculo alcanza cada modelo.

In [ ]:
orden_m = ["Popularidad", "Pop. por segmento + reglas", "Contenido (TF-IDF)", "KNN usuario", "KNN ítem", "LightFM híbrido", "LightFM + reglas", "Oráculo"]
def resumen(esq, col="ndcg"):
    d = resultados[resultados.esquema == esq].copy()
    ora = d[d.modelo == "Oráculo"].set_index(["escenario", "replica"])["ndcg"]
    d["pct_techo"] = 100 * d["ndcg"] / d.set_index(["escenario", "replica"]).index.map(ora).values
    g = d.groupby(["modelo", "escenario"])[col if col != "pct_techo" else "pct_techo"].agg(["mean", "std"]).unstack("escenario")
    fmt = (lambda m, s: f"{m:.3f} ± {s:.3f}") if col != "pct_techo" else (lambda m, s: f"{m:.0f} % ± {s:.0f}")
    tab = pd.DataFrame({e: [fmt(g.loc[m, ("mean", e)], g.loc[m, ("std", e)]) for m in orden_m] for e in ESC}, index=orden_m)
    return tab, g
tab_ndcg, g_ndcg = resumen("global", "ndcg")
tab_techo, g_techo = resumen("global", "pct_techo")
print("NDCG@10 · corte global · media ± desviación entre 10 réplicas"); display(tab_ndcg)
print("% del techo del oráculo · corte global"); display(tab_techo.drop(index="Oráculo"))
print("Estudiantes evaluables por réplica (corte global):"); display(resultados[(resultados.esquema == "global") & (resultados.modelo == "Popularidad")].groupby("escenario").evaluables.agg(["mean", "min", "max"]).round(0))
tab_ndcg.to_csv(RES / "replicas_ndcg_global.csv"); tab_techo.to_csv(RES / "replicas_pct_techo_global.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
modelos_g = ["Popularidad", "Pop. por segmento + reglas", "Contenido (TF-IDF)", "KNN usuario", "KNN ítem", "LightFM + reglas"]
cols_g = [GRIS, NARANJA, AMARILLO, AQUA, MAGENTA, AZUL]
w = 0.13; x = np.arange(len(ESC))
for j, (m, c) in enumerate(zip(modelos_g, cols_g)):
    mu = [g_techo.loc[m, ("mean", e)] for e in ESC]; sd = [g_techo.loc[m, ("std", e)] for e in ESC]
    ax.bar(x + (j - 2.5) * w, mu, width=w - 0.01, yerr=sd, color=c, label=m, capsize=2, error_kw={"lw": 1})
ax.set_xticks(x, list(ESC), fontsize=8); ax.set_ylabel("% del techo del oráculo (NDCG@10)")
ax.set_title("Corte global: desempeño relativo por escenario (media ± sd de 10 réplicas)")
ax.legend(frameon=False, fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.18)); ax.grid(axis="x", visible=False)
guardar(fig, "23_escenarios_pct_techo"); plt.show()

**Interpretación.** *(NDCG@10 medio de 10 réplicas; la desviación entre réplicas es de 0.013 a 0.029, del mismo orden que las diferencias entre los mejores modelos).*

- **No existe un modelo ganador en todos los escenarios.**
- **E0 (reglas del v1):** LightFM + reglas 0.219, Popularidad por segmento + reglas 0.211, popularidad global 0.168, techo 0.291. Es el mismo patrón del notebook 04 con una muestra mayor: LightFM llega al 75 % del techo y el baseline por segmento al 73 %.
- **E1 (perfil débil), E2 (popularidad dominante) y E5 (embudo bajo):** LightFM + reglas y el baseline por segmento quedan prácticamente empatados (E1: 0.139 y 0.147; E2: 0.214 y 0.218; E5: 0.158 y 0.158), y la popularidad global se acerca (0.133, 0.203 y 0.123).
- **E3 (ruta secuencial):** LightFM + reglas 0.196 y baseline por segmento 0.195. Los métodos basados en historial no ganan: KNN usuario 0.158 y contenido 0.114.
- **E4 (gustos latentes):** **el orden se invierte.** Contenido 0.226, KNN ítem 0.224 y KNN usuario 0.212 superan con claridad a LightFM + reglas (0.154), al baseline por segmento (0.106) y a la popularidad (0.102). El techo sube a 0.415 porque los gustos individuales hacen la elección más predecible para quien los conoce.
- **Fracción del techo de LightFM + reglas** según el escenario: de 37 % (E4) a 78 % (E2).

## 4. ¿Dónde gana LightFM y dónde gana la personalización simple?

Para cada escenario y réplica se calcula la diferencia de NDCG@10 (media sobre estudiantes) entre pares de modelos y se cuenta en **cuántas de las 10 réplicas** el primero supera al segundo, y en cuántas la diferencia es significativa con Wilcoxon al 5 % (descriptivo, **sin corrección por comparaciones múltiples**: es un conteo de réplicas, no una prueba global).

In [ ]:
d = diferencias[diferencias.esquema == "global"]
def conteo(df):
    return pd.Series({"dif. media": df["dif"].mean(), "sd entre réplicas": df["dif"].std(),
                      "réplicas donde gana (de 10)": int((df["dif"] > 0).sum()),
                      "réplicas con p < 0.05 a favor": int(((df["dif"] > 0) & (df["p"] < 0.05)).sum()),
                      "réplicas con p < 0.05 en contra": int(((df["dif"] < 0) & (df["p"] < 0.05)).sum())})
tabla_dif = d.groupby(["comparacion", "escenario"]).apply(conteo)
tabla_dif.to_csv(RES / "replicas_comparaciones_global.csv")
for comp in d.comparacion.unique():
    print(comp); display(tabla_dif.loc[comp].round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=False)
for ax, comp, tit in zip(axes, ["LightFM + reglas − Pop. por segmento + reglas", "LightFM + reglas − Popularidad"],
                         ["LightFM + reglas menos Pop. por segmento + reglas", "LightFM + reglas menos Popularidad global"]):
    dd = d[d.comparacion == comp]
    for k, e in enumerate(ESC):
        v = dd[dd.escenario == e]["dif"].values
        ax.scatter(np.full(len(v), k) + np.linspace(-0.15, 0.15, len(v)), v, color=AZUL, s=18, alpha=0.8)
        ax.hlines(v.mean(), k - 0.25, k + 0.25, color=NARANJA, lw=2)
    ax.axhline(0, color=GRIS, lw=1); ax.set_xticks(range(len(ESC)), list(ESC), fontsize=7, rotation=20, ha="right")
    ax.set_title(tit, fontsize=10); ax.set_ylabel("diferencia de NDCG@10 por réplica"); ax.grid(axis="x", visible=False)
guardar(fig, "24_diferencias_por_replica"); plt.show()

**Interpretación** *(conteo de réplicas, sin corrección por comparaciones múltiples: es descriptivo)*.

**LightFM + reglas frente a la popularidad global.** Diferencia media de NDCG@10: +0.051 (E0), +0.006 (E1), +0.010 (E2), +0.021 (E3), +0.052 (E4) y +0.035 (E5). Gana en 10 de 10 réplicas en E0, E4 y E5, en 9 de 10 en E3, 7 de 10 en E2 y 6 de 10 en E1. Es significativo al 5 % dentro de la réplica en 8, 0, 2, 5, 8 y 3 réplicas respectivamente, y **nunca** es significativamente peor. La conclusión del v1 ("supera a la popularidad global") es robusta en dirección, pero su tamaño depende del escenario: cuando el perfil explica poco (E1) o domina la popularidad (E2) la ventaja casi desaparece.

**LightFM + reglas frente a la popularidad por segmento + reglas.** Diferencia media: +0.007 (E0), −0.008 (E1), −0.004 (E2), 0.000 (E3), **+0.048 (E4)** y 0.000 (E5). Solo en E4 gana de forma consistente (10 de 10 réplicas; 8 significativas). En los demás escenarios gana en 1 a 8 réplicas de 10 y casi ninguna diferencia es significativa; hay 1 réplica significativamente en contra en E1, E2 y E3. Esto confirma lo visto en el notebook 04: **frente a una personalización simple por objetivo laboral, la ventaja de LightFM no es robusta.**

**Popularidad por segmento frente a la global.** Gana en 10, 7, 8, 9, 6 y 10 réplicas de 10 (E0 a E5): personalizar por objetivo laboral ayuda de forma bastante consistente, aunque sea poco.

### Resumen de robustez de la elección del modelo

Para cada modelo: promedio y **peor caso** del porcentaje del techo (corte global) entre los 6 escenarios, y en cuántos escenarios queda primero o empatado con el primero (diferencia menor que 0.01 de NDCG@10 respecto del mejor).

In [ ]:
g = resultados[resultados.esquema == "global"].copy()
ora = g[g.modelo == "Oráculo"].set_index(["escenario", "replica"])["ndcg"]
g["pct"] = 100 * g["ndcg"] / g.set_index(["escenario", "replica"]).index.map(ora).values
pe = g[g.modelo != "Oráculo"].groupby(["modelo", "escenario"])[["pct", "ndcg"]].mean().reset_index()
mejor = pe.groupby("escenario")["ndcg"].max()
pe["cerca_del_mejor"] = pe.apply(lambda r: r.ndcg >= mejor[r.escenario] - 0.01, axis=1)
rob = pe.groupby("modelo").agg(**{"% del techo (promedio)": ("pct", "mean"), "% del techo (peor escenario)": ("pct", "min"),
                                  "escenarios en que es el mejor o empata (de 6)": ("cerca_del_mejor", "sum")})
rob = rob.sort_values("% del techo (promedio)", ascending=False)
rob.to_csv(RES / "replicas_robustez_eleccion.csv")
display(rob.round(1))

## 5. Fuga entre usuarios: esquema por usuario frente a corte global

En el notebook 01 se aceptó el esquema por usuario (más estudiantes evaluables) a costa de una fuga entre usuarios. Aquí se mide **cuánto distorsiona**, comparando ambos esquemas **sobre los mismos datasets**.

In [ ]:
c = resultados.pivot_table(index=["escenario", "replica", "modelo"], columns="esquema", values="ndcg").reset_index()
c["dif (usuario − global)"] = c["usuario"] - c["global"]
efecto = c.groupby("modelo")[["global", "usuario", "dif (usuario − global)"]].mean().reindex(orden_m)
efecto["% de cambio"] = 100 * efecto["dif (usuario − global)"] / efecto["global"]
display(efecto.round(3))
# ¿cambia el orden de los modelos?
rk = resultados[resultados.modelo != "Oráculo"].copy()
rk["rank"] = rk.groupby(["escenario", "esquema", "replica"])["ndcg"].rank(ascending=False)
rho = []
for (e, r), g in rk.groupby(["escenario", "replica"]):
    a = g[g.esquema == "global"].set_index("modelo")["rank"]; b = g[g.esquema == "usuario"].set_index("modelo")["rank"].reindex(a.index)
    rho.append({"escenario": e, "replica": r, "spearman": stats.spearmanr(a, b)[0]})
rho = pd.DataFrame(rho)
print("Correlación de Spearman entre el orden de modelos con corte global y con corte por usuario (media por escenario):")
display(rho.groupby("escenario")["spearman"].agg(["mean", "min"]).round(2))
efecto.to_csv(RES / "replicas_efecto_fuga.csv")

**Interpretación** *(promedio de los 6 escenarios y 10 réplicas)*.
- Con el corte **por usuario** el NDCG@10 sale **entre 6 % y 10 % más bajo** que con el corte global en todos los modelos salvo el oráculo (por ejemplo, LightFM: 0.182 con corte global y 0.169 por usuario; popularidad: 0.151 y 0.138). Es decir, la fuga entre usuarios **no infla** los resultados; las poblaciones evaluadas son distintas (más estudiantes con el corte por usuario), así que no hay que sobreinterpretar el signo.
- La ventaja de LightFM sobre la popularidad es la **misma** con ambos esquemas (+0.031 en los dos).
- El orden de los modelos se mantiene: correlación de Spearman media entre ambos esquemas de 0.84 a 0.94 según el escenario (mínimo de 0.61 en una réplica aislada).

Esto respalda haber usado el esquema por usuario en el v1: con estos datos, la fuga que se aceptó no cambia las conclusiones. Se puede decir con evidencia y no solo con la prueba de robustez de 30 estudiantes del notebook 03.

## 6. Conclusiones

**Lo que sí es robusto (se mantiene en los 6 escenarios)**
1. LightFM + reglas supera en promedio a la popularidad global en todos los escenarios, nunca queda significativamente por debajo y tiene el **mayor promedio del % del techo (62.6 %)** y el mejor peor caso entre los modelos prácticos que se comparan (37 %, frente a 25 % de la popularidad por segmento y 25 % de la global). Es el mejor o empata con el mejor en 5 de 6 escenarios (todos menos E4).
2. Personalizar por objetivo laboral mejora a la popularidad global en casi todas las réplicas.
3. El esquema por usuario no distorsiona el orden de los modelos ni la ventaja de LightFM.

**Lo que no es robusto**
4. "LightFM supera a una personalización simple": solo ocurre cuando existen gustos individuales no observables (E4). En los otros 5 escenarios empata con la popularidad por segmento + reglas (diferencias de −0.008 a +0.007).
5. "Los métodos de contenido y KNN son inferiores": en E4 son los **mejores** (0.21 a 0.23 frente a 0.154 de LightFM). Son malos en el v1 y en casi todos los escenarios precisamente porque el simulador del v1 se genera a partir del perfil.
6. Cualquier cifra absoluta: el desempeño de LightFM + reglas va de 37 % a 78 % del techo según los supuestos.

**Cómo conviene decirlo ante el jurado**
- *"Evaluamos el sistema bajo 6 supuestos distintos del simulador, con 10 datasets independientes por escenario. LightFM + reglas es el modelo con mejor desempeño promedio y mejor peor caso, y supera a la popularidad global en todos los escenarios; pero frente a una personalización simple por objetivo laboral no hay ventaja robusta, y si los estudiantes reales tienen gustos individuales no capturados por su perfil, los métodos basados en historial pueden rendir más."*
- Reconocer como **limitación central** que el v1 corresponde a un mundo dirigido por el perfil (E0). Con estudiantes reales, la realidad probablemente esté entre E0 y E4, y ahí convendría un modelo que use el historial con más peso (por ejemplo, combinar LightFM con contenido o KNN). Eso **no se probó** aquí y es una mejora futura, no un resultado.
- Los hallazgos del notebook 06 son sobre el simulador, no sobre estudiantes reales: lo que aporta es saber **bajo qué supuestos** se sostiene cada afirmación.